# Setup and load all models from models/ folder

In [1]:
import pickle
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"

model_names = ["RW", "DNS", "Ridge", "XGBoost"]

model_preds = {}
model_actuals = {}
model_metrics = {}

for name in model_names:
    path = MODEL_DIR / f"{name}.pkl"
    with open(path, "rb") as f:
        bundle = pickle.load(f)

    print(f"Loaded {name} from {path}, keys: {list(bundle.keys())}")

    model_preds[name]   = bundle["predictions"]
    model_actuals[name] = bundle["actuals"]
    model_metrics[name] = bundle["metrics"]   # <--- NEW

print("Models loaded:", list(model_preds.keys()))


Loaded RW from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/RW.pkl, keys: ['predictions', 'actuals', 'metrics']
Loaded DNS from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/DNS.pkl, keys: ['predictions', 'actuals', 'metrics']
Loaded Ridge from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/Ridge.pkl, keys: ['predictions', 'actuals', 'metrics', 'hyperparameters']
Loaded XGBoost from /Users/chriss/Desktop/Studium/Semester/7. Semester/Bachelorarbeit/yield-curve-forecasting/yield-curve-forecasting/models/XGBoost.pkl, keys: ['predictions', 'actuals', 'metrics', 'hyperparameters']
Models loaded: ['RW', 'DNS', 'Ridge', 'XGBoost']


In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

# ----------------- Paths -----------------
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# ----------------- Load raw FRED data -----------------
DGS1 = pd.read_csv(DATA_DIR / "DGS1.csv")
DGS2 = pd.read_csv(DATA_DIR / "DGS2.csv")
DGS5 = pd.read_csv(DATA_DIR / "DGS5.csv")
DGS10 = pd.read_csv(DATA_DIR / "DGS10.csv")

# 2y, 5y, 10y panel
merged = (
    DGS2.merge(DGS5, on="observation_date", how="inner")
        .merge(DGS10, on="observation_date", how="inner")
)
merged = merged.dropna(subset=["DGS2", "DGS5", "DGS10"])
merged = merged.set_index("observation_date")
merged.index = pd.to_datetime(merged.index, errors="coerce")

# Short rate (1y) as separate series
DGS1 = DGS1.dropna(subset=["DGS1"])
DGS1 = DGS1.set_index("observation_date")
DGS1.index = pd.to_datetime(DGS1.index, errors="coerce")

short_rate = DGS1["DGS1"]   # <-- THIS is what you pass to econ functions

# ----------------- Settings -----------------
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons = [1, 5, 10, 30]
idx = merged.index
train_start = pd.Timestamp("2009-01-02")
train_end   = pd.Timestamp("2018-12-31")

test_start  = pd.Timestamp("2019-01-02")
test_end    = pd.Timestamp("2025-11-25")  


# Diebold-Marino Test

In [2]:
import numpy as np
from scipy.stats import norm

def dm_test(errors_model1, errors_model2, h=1, lag=None):
    """
    Diebold-Mariano test for equal predictive accuracy (squared-error loss).

    Parameters
    ----------
    errors_model1 : array-like
        Forecast errors of model 1 (y - y_hat1), length N.
    errors_model2 : array-like
        Forecast errors of model 2 (y - y_hat2), length N.
    h : int, optional
        Forecast horizon (in periods). Used only if lag is None.
    lag : int, optional
        Newey-West truncation lag for HAC variance.
        If None, lag is set to max(h-1, 0).

    Returns
    -------
    dm_stat : float
        Diebold-Mariano test statistic.
    p_value : float
        Two-sided p-value under asymptotic N(0,1).
    """

    e1 = np.asarray(errors_model1)
    e2 = np.asarray(errors_model2)

    # Drop NaNs in parallel
    mask = np.isfinite(e1) & np.isfinite(e2)
    e1 = e1[mask]
    e2 = e2[mask]

    if e1.shape != e2.shape:
        raise ValueError("Error series must have the same length after NaN removal.")

    N = len(e1)
    if N < 5:
        raise ValueError("Not enough observations for DM test.")

    # Squared-error loss
    L1 = e1 ** 2
    L2 = e2 ** 2

    # Loss differential
    d = L1 - L2
    d_bar = np.mean(d)

    # Newey–West HAC variance of d_t
    if lag is None:
        lag = max(h - 1, 0)

    d_centered = d - d_bar
    gamma0 = np.dot(d_centered, d_centered) / N
    var_d = gamma0

    for k in range(1, lag + 1):
        cov = np.dot(d_centered[k:], d_centered[:-k]) / N
        weight = 1.0 - k / (lag + 1)  # Bartlett weight
        var_d += 2.0 * weight * cov

    dm_stat = d_bar / np.sqrt(var_d / N)
    p_value = 2 * (1 - norm.cdf(np.abs(dm_stat)))

    return dm_stat, p_value


In [6]:
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons = [1, 5, 10, 30]

dm_results = []

benchmark_name = "RW"

if benchmark_name not in model_actuals:
    raise ValueError(f"Benchmark model '{benchmark_name}' not loaded.")

# Use RW's actual series as canonical ground truth
rw_actual_series = model_actuals[benchmark_name]

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        # Use RW's actual series as canonical ground truth for this (maturity, h)
        if key not in rw_actual_series:
            continue

        actual = rw_actual_series[key].copy()
        actual.name = "actual"

        # Build a dict of aligned prediction series for all models that exist
        preds_for_key = {}
        for model_name, pred_dict in model_preds.items():
            if key not in pred_dict:
                continue
            preds_for_key[model_name] = pred_dict[key].rename(model_name)

        # Need at least RW + one other model with forecasts
        if benchmark_name not in preds_for_key or len(preds_for_key) < 2:
            continue

        # Compare each non-RW model against RW
        for model_name, pred_series in preds_for_key.items():
            if model_name == benchmark_name:
                continue

            # Pairwise alignment: actual, RW, and the other model
            df_pair = pd.concat(
                [
                    actual,
                    preds_for_key[benchmark_name],  # RW predictions
                    pred_series                     # other model predictions
                ],
                axis=1,
                join="inner"
            ).dropna()

            if df_pair.empty:
                continue

            # Forecast errors: e = actual - prediction
            err_rw    = df_pair["actual"] - df_pair[benchmark_name]
            err_other = df_pair["actual"] - df_pair[model_name]

            # Diebold–Mariano test (RW as Model 1, other model as Model 2)
            dm_stat, p_val = dm_test(err_rw.values, err_other.values, h=h)

            dm_results.append({
                "Maturity": maturity,
                "Horizon":  h,
                "Model_1":  benchmark_name,
                "Model_2":  model_name,
                "DM_stat":  dm_stat,
                "p_value":  p_val,
                "N":        len(df_pair)
            })

dm_results_df = pd.DataFrame(dm_results)
dm_results_df = dm_results_df.sort_values(["Maturity", "Horizon", "Model_2"])

display(dm_results_df)

,Maturity,Horizon,Model_1,Model_2,DM_stat,p_value,N
24,DGS10,1,RW,DNS,-2.076076,0.037887,1725
25,DGS10,1,RW,Ridge,-1.350475,0.176864,1725
26,DGS10,1,RW,XGBoost,-3.805750,0.000141,1712
27,DGS10,5,RW,DNS,-2.542185,0.011016,1721
28,DGS10,5,RW,Ridge,-1.547944,0.121636,1721
29,DGS10,5,RW,XGBoost,-3.977064,0.000070,1708
30,DGS10,10,RW,DNS,-2.605574,0.009172,1716
31,DGS10,10,RW,Ridge,-2.768207,0.005637,1716
32,DGS10,10,RW,XGBoost,-2.632533,0.008475,1703
33,DGS10,30,RW,DNS,-2.251019,0.024384,1696


In [17]:
dm_results_df[dm_results_df["Model_2"] == "Ridge"]

,Maturity,Horizon,Model_1,Model_2,DM_stat,p_value,N
25,DGS10,1,RW,Ridge,-1.350475,0.176864,1725
28,DGS10,5,RW,Ridge,-1.547944,0.121636,1721
31,DGS10,10,RW,Ridge,-2.768207,0.005637,1716
34,DGS10,30,RW,Ridge,-3.158788,0.001584,1696
1,DGS2,1,RW,Ridge,-1.312140,0.189473,1725
4,DGS2,5,RW,Ridge,-1.378059,0.168185,1721
7,DGS2,10,RW,Ridge,-1.521352,0.128172,1716
10,DGS2,30,RW,Ridge,-2.584080,0.009764,1696
13,DGS5,1,RW,Ridge,-1.506511,0.131936,1725
16,DGS5,5,RW,Ridge,-2.479280,0.013165,1721


In [11]:
maturity_cols = ["DGS2", "DGS5", "DGS10"]
horizons      = [1, 5, 10, 30]

dm_results = []

# -------------------------------------------------
# Benchmark: DNS
# -------------------------------------------------
benchmark_name = "DNS"
models_to_compare = ["Ridge", "XGBoost"]   # you can add "RW" here if you change your mind

if benchmark_name not in model_actuals:
    raise ValueError(f"Benchmark model '{benchmark_name}' not loaded.")

# Use DNS' actual series as canonical ground truth (should be identical across models)
dns_actual_series = model_actuals[benchmark_name]

for maturity in maturity_cols:
    for h in horizons:
        key = (maturity, h)

        # Check that we have actuals for this maturity/horizon
        if key not in dns_actual_series:
            continue

        # Actual LEVEL yields at forecast dates
        actual = dns_actual_series[key].copy()
        actual.name = "actual"

        # -------------------------------------------------
        # Collect prediction series for this (maturity, h)
        # -------------------------------------------------
        preds_for_key = {}
        for model_name, pred_dict in model_preds.items():
            if key not in pred_dict:
                continue
            preds_for_key[model_name] = pred_dict[key].rename(model_name)

        # Need benchmark + at least one comparison model
        if benchmark_name not in preds_for_key:
            continue

        # -------------------------------------------------
        # Compare each model in models_to_compare vs DNS
        # -------------------------------------------------
        for model_name in models_to_compare:
            if model_name not in preds_for_key:
                continue
            if model_name == benchmark_name:
                continue

            pred_bench = preds_for_key[benchmark_name]
            pred_other = preds_for_key[model_name]

            # Pairwise alignment: actual, DNS, and the other model
            df_pair = pd.concat(
                [actual, pred_bench, pred_other],
                axis=1,
                join="inner"
            ).dropna()

            if df_pair.empty:
                continue

            # Forecast errors: e = actual - prediction
            err_bench = df_pair["actual"] - df_pair[benchmark_name]   # DNS errors
            err_other = df_pair["actual"] - df_pair[model_name]       # other model

            # Diebold–Mariano test (DNS as Model 1, other model as Model 2)
            dm_stat, p_val = dm_test(err_bench.values, err_other.values, h=h)

            dm_results.append({
                "Maturity": maturity,
                "Horizon":  h,
                "Model_1":  benchmark_name,
                "Model_2":  model_name,
                "DM_stat":  dm_stat,
                "p_value":  p_val,
                "N":        len(df_pair),
            })

# -------------------------------------------------
# Build results DataFrame
# -------------------------------------------------
dm_dns_benchmark_df = (
    pd.DataFrame(dm_results)
    .sort_values(["Maturity", "Horizon", "Model_2"])
    .reset_index(drop=True)
)

display(dm_dns_benchmark_df)


,Maturity,Horizon,Model_1,Model_2,DM_stat,p_value,N
0,DGS10,1,DNS,Ridge,-0.624639,0.532208,1725
1,DGS10,1,DNS,XGBoost,-3.686500,0.000227,1712
2,DGS10,5,DNS,Ridge,0.764102,0.444806,1721
3,DGS10,5,DNS,XGBoost,-3.672787,0.000240,1708
4,DGS10,10,DNS,Ridge,-0.366486,0.714003,1716
5,DGS10,10,DNS,XGBoost,-2.172855,0.029791,1703
6,DGS10,30,DNS,Ridge,-2.741397,0.006118,1696
7,DGS10,30,DNS,XGBoost,-2.883782,0.003929,1683
8,DGS2,1,DNS,Ridge,-0.937815,0.348339,1725
9,DGS2,1,DNS,XGBoost,-3.879112,0.000105,1712


# Root-Mean-Squared Error

In [11]:
# ----------------------------------------------------
# RMSE comparison table across models and horizons
# ----------------------------------------------------

rmse_tables = []

for model_name, df in model_metrics.items():
    if "RMSE" not in df.columns:
        print(f"⚠️ No RMSE column found for {model_name}, skipping.")
        continue

    rmse_tables.append(
        df[["RMSE"]].rename(columns={"RMSE": model_name})
    )

rmse_compare_df = pd.concat(rmse_tables, axis=1).sort_index()

display(rmse_compare_df)


RW       DNS     Ridge   XGBoost
Maturity Horizon                                        
DGS10    1        0.059004  0.059081  0.059137  0.061816
         5        0.126819  0.127669  0.127458  0.136784
         10       0.179402  0.181670  0.181782  0.193798
         30       0.328191  0.338187  0.346037  0.388307
DGS2     1        0.061250  0.061312  0.061421  0.065850
         5        0.131312  0.131994  0.132197  0.144119
         10       0.188868  0.190579  0.190964  0.210959
         30       0.359041  0.366263  0.369247  0.413400
DGS5     1        0.062598  0.062685  0.062806  0.065312
         5        0.134188  0.135193  0.135146  0.141190
         10       0.190236  0.192971  0.193134  0.206624
         30       0.349621  0.361938  0.364119  0.414612

In [7]:
import pandas as pd
import numpy as np

def audit_alignment_for_key(
    maturity: str,
    horizon: int,
    model_preds: dict,
    model_actuals: dict,
    merged: pd.DataFrame,
    models=("RW", "DNS", "Ridge", "XGBoost"),
    canonical_actual_model="RW",
    print_head_tail=3,
):
    """
    Audits alignment for a single (maturity, horizon):
    - checks if all series are indexed on the SAME forecast dates
    - checks if 'actual' equals merged[maturity] on those dates
    - shows how many observations survive pairwise joins
    - flags if any model appears indexed by origin dates instead of forecast dates
    """

    key = (maturity, horizon)

    # --- 1) collect actual series (canonical) ---
    if canonical_actual_model not in model_actuals or key not in model_actuals[canonical_actual_model]:
        raise ValueError(f"Missing canonical actuals: {canonical_actual_model} for {key}")

    actual = model_actuals[canonical_actual_model][key].copy()
    actual.name = "actual"

    # --- 2) sanity check: actual should match merged on same index ---
    merged_actual = merged.loc[actual.index, maturity].copy()
    merged_actual.name = "merged_actual"

    diff = (actual - merged_actual).dropna()
    max_abs_diff = diff.abs().max() if len(diff) else 0.0

    print(f"\n=== ALIGNMENT AUDIT | {maturity} | h={horizon} ===")
    print(f"Canonical actual model: {canonical_actual_model}")
    print(f"Actual index range: {actual.index.min().date()} -> {actual.index.max().date()} (N={len(actual)})")
    print(f"Check actual == merged[{maturity}] on same dates: max|diff| = {max_abs_diff:.12f}")

    if max_abs_diff > 1e-9:
        print("⚠️  WARNING: 'actual' series does NOT match merged on its index. This indicates misalignment or wrong actuals.")
    else:
        print("✅  Actual matches merged on same dates.")

    # --- 3) collect predictions per model ---
    preds = {}
    for m in models:
        s = model_preds.get(m, {}).get(key, None)
        if s is None:
            print(f"⚠️  Missing predictions for model {m} at {key}")
            continue
        preds[m] = s.rename(m)

    if len(preds) < 2:
        raise ValueError("Need at least two models with predictions to audit alignment meaningfully.")

    # --- 4) check index stats & overlaps ---
    for m, s in preds.items():
        print(f"{m:7s} pred index: {s.index.min().date()} -> {s.index.max().date()} (N={len(s)})")

    # Common intersection across all available predictions + actual
    idx_common_all = actual.index
    for s in preds.values():
        idx_common_all = idx_common_all.intersection(s.index)

    print(f"\nCommon dates across actual + all available preds: N={len(idx_common_all)}")
    if len(idx_common_all) == 0:
        print("❌  No common dates. This almost always means at least one model is indexed differently (origin vs forecast).")

    # Show a few example dates and whether each model has them
    sample_dates = list(idx_common_all[:print_head_tail]) + list(idx_common_all[-print_head_tail:]) if len(idx_common_all) else []
    if sample_dates:
        print("\nSample common dates (head/tail):")
        for d in sample_dates:
            flags = " | ".join([f"{m}:{'✓' if d in preds[m].index else 'x'}" for m in preds.keys()])
            print(f"  {d.date()} -> {flags}")

    # --- 5) pairwise alignment diagnostics (like DM uses) ---
    print("\nPairwise join sizes (actual + modelA + modelB):")
    pred_items = list(preds.items())
    for i in range(len(pred_items)):
        for j in range(i + 1, len(pred_items)):
            m1, s1 = pred_items[i]
            m2, s2 = pred_items[j]
            df_pair = pd.concat([actual, s1, s2], axis=1, join="inner").dropna()
            print(f"  {m1:7s} vs {m2:7s}: N={len(df_pair)} "
                  f"| {df_pair.index.min().date() if len(df_pair) else 'NA'} -> {df_pair.index.max().date() if len(df_pair) else 'NA'}")

    # --- 6) origin-vs-forecast indexing heuristic ---
    # If series is indexed by origin t instead of forecast t+h, then shifting its index forward by h
    # should increase overlap with actual (which should be on forecast dates).
    print("\nOrigin-vs-forecast index heuristic (does shifting index by +h increase overlap with actual?):")
    for m, s in preds.items():
        overlap_now = len(actual.index.intersection(s.index))
        s_shifted = s.copy()
        s_shifted.index = s_shifted.index.shift(horizon, freq="B")  # business-day shift
        overlap_shifted = len(actual.index.intersection(s_shifted.index))

        print(f"  {m:7s}: overlap_now={overlap_now}, overlap_if_index_shifted_by_+{horizon}B={overlap_shifted}")

        if overlap_shifted > overlap_now + 5:  # threshold to avoid noise
            print(f"    ⚠️  {m} looks like it might be indexed on ORIGIN dates t (not forecast dates t+h).")

    print("\n=== END AUDIT ===\n")


In [10]:
audit_alignment_for_key(
    maturity="DGS10",
    horizon=10,
    model_preds=model_preds,
    model_actuals=model_actuals,
    merged=merged,
    models=("RW", "DNS", "Ridge", "XGBoost"),
    canonical_actual_model="RW",
)



=== ALIGNMENT AUDIT | DGS10 | h=10 ===
Canonical actual model: RW
Actual index range: 2019-01-02 -> 2025-11-25 (N=1726)
Check actual == merged[DGS10] on same dates: max|diff| = 0.000000000000
✅  Actual matches merged on same dates.
RW      pred index: 2019-01-02 -> 2025-11-25 (N=1726)
DNS     pred index: 2019-01-16 -> 2025-11-25 (N=1716)
Ridge   pred index: 2019-01-16 -> 2025-11-25 (N=1716)
XGBoost pred index: 2019-01-16 -> 2025-11-05 (N=1703)

Common dates across actual + all available preds: N=1703

Sample common dates (head/tail):
  2019-01-16 -> RW:✓ | DNS:✓ | Ridge:✓ | XGBoost:✓
  2019-01-17 -> RW:✓ | DNS:✓ | Ridge:✓ | XGBoost:✓
  2019-01-18 -> RW:✓ | DNS:✓ | Ridge:✓ | XGBoost:✓
  2025-11-03 -> RW:✓ | DNS:✓ | Ridge:✓ | XGBoost:✓
  2025-11-04 -> RW:✓ | DNS:✓ | Ridge:✓ | XGBoost:✓
  2025-11-05 -> RW:✓ | DNS:✓ | Ridge:✓ | XGBoost:✓

Pairwise join sizes (actual + modelA + modelB):
  RW      vs DNS    : N=1716 | 2019-01-16 -> 2025-11-25
  RW      vs Ridge  : N=1716 | 2019-01-16 -> 202